# TF-IDF + Logistic Regression + sublinear_tf=True

Para as tentativas de melhoria, foi utilizado o conjunto de dados limpo e variação do ngrams_range (experimentos A, B, C, D e F).

In [1]:
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

## 1. Carregamento dos dados

In [3]:
df = pd.read_csv("../data/train_limpo.csv") # Conjunto de treino limpo

Y = df["clarity"]

print(df.shape)
df.head()

(20092, 2)


,resp_text,clarity
0,prezado a senhor a esclarecemos que o serviço ...,c5
1,prezada cidadã as informações sobre óbitos no ...,c1
2,prezado senhor julio a ouvidoria geral da petr...,c1
3,prezado a senhor a esclarecemos que o serviço ...,c234
4,senhor o serviço de informações ao cidadão do ...,c234


## 2. Separação dos dados

In [4]:
x_train_text, x_test_text, y_train, y_test = train_test_split(
    df["resp_text"],
    Y,
    test_size=0.2,
    random_state=123,
    stratify=Y
)

print("Treino:", x_train_text.shape)
print("Teste:", x_test_text.shape)

Treino: (16073,)
Teste: (4019,)


## 3. Vetorização TF-IDF

Neste experimento, utilizamos `sublinear_tf=True`, que aplica uma ponderação sublinear à frequência dos termos.

Para as tentativas de melhoria, tentei variar o ngram_range para (1, 4) (experimento B) e (1, 5) (experimento C).

Além disso, também tentei utilizar n-gramas de caracteres com range de (2, 5) (experimento D).

In [5]:
vect = TfidfVectorizer(
    analyzer="word",
    sublinear_tf=True,
    ngram_range=(1, 3)
)

x_train = vect.fit_transform(x_train_text)
x_test = vect.transform(x_test_text)

print("Treino TF-IDF:", x_train.shape)
print("Teste TF-IDF:", x_test.shape)

Treino TF-IDF: (16073, 1196293)
Teste TF-IDF: (4019, 1196293)


## 4. Regressão Logística

In [7]:
clf = LogisticRegression(
    C=1,
    class_weight="balanced",
    max_iter=1000
)

clf.fit(
    x_train,
    y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


## 5. Avaliação

In [8]:
predicted = clf.predict(x_test)

score = f1_score(
    y_test,
    predicted,
    average="macro"
)

accuracy = accuracy_score(
    y_test,
    predicted
)

print(f"Macro F1: {score:.4f}")
print(f"Accuracy: {accuracy:.4f}")

Macro F1: 0.4799
Accuracy: 0.4800


In [9]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        predicted
    )
)

              precision    recall  f1-score   support

          c1       0.51      0.52      0.51      1269
        c234       0.41      0.39      0.40      1371
          c5       0.52      0.53      0.52      1379

    accuracy                           0.48      4019
   macro avg       0.48      0.48      0.48      4019
weighted avg       0.48      0.48      0.48      4019



In [ ]:
# # ============================================================
# # VALIDAÇÃO CRUZADA — TF-IDF + Logistic Regression
# # Melhor configuração encontrada
# # ============================================================

# import pandas as pd
# import numpy as np

# from sklearn.pipeline import Pipeline
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import StratifiedKFold, cross_validate


# # ------------------------------------------------------------
# # 1. Dados
# # ------------------------------------------------------------

# df = pd.read_csv("train_limpo.csv")

# X = df["resp_text"]
# y = df["clarity"]


# # ------------------------------------------------------------
# # 2. Melhor configuração encontrada
# # ------------------------------------------------------------

# pipeline = Pipeline([
#     (
#         "tfidf",
#         TfidfVectorizer(
#             analyzer="word",
#             sublinear_tf=True,
#             ngram_range=(1, 3)
#         )
#     ),
#     (
#         "logreg",
#         LogisticRegression(
#             C=1.0,
#             class_weight="balanced",
#             max_iter=1000
#         )
#     )
# ])


# # ------------------------------------------------------------
# # 3. Cross-validation estratificada
# # ------------------------------------------------------------

# cv = StratifiedKFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=123
# )


# # ------------------------------------------------------------
# # 4. Avaliação
# # ------------------------------------------------------------

# results = cross_validate(
#     pipeline,
#     X,
#     y,
#     cv=cv,
#     scoring={
#         "f1_macro": "f1_macro",
#         "accuracy": "accuracy"
#     },
#     return_train_score=True,
#     n_jobs=-1
# )


# # ------------------------------------------------------------
# # 5. Resultados por fold
# # ------------------------------------------------------------

# print("Resultados por fold:\n")

# for i in range(5):
#     print(
#         f"Fold {i+1}: "
#         f"F1 Macro = {results['test_f1_macro'][i]:.4f} | "
#         f"Accuracy = {results['test_accuracy'][i]:.4f}"
#     )


# # ------------------------------------------------------------
# # 6. Média e desvio padrão
# # ------------------------------------------------------------

# print("\nResultado da validação cruzada:")

# print(
#     f"Macro F1: "
#     f"{results['test_f1_macro'].mean():.4f} "
#     f"± {results['test_f1_macro'].std():.4f}"
# )

# print(
#     f"Accuracy: "
#     f"{results['test_accuracy'].mean():.4f} "
#     f"± {results['test_accuracy'].std():.4f}"
# )

# print("\nTreino:")
# print(
#     f"Macro F1: "
#     f"{results['train_f1_macro'].mean():.4f} "
#     f"± {results['train_f1_macro'].std():.4f}"
# )

# print(
#     f"Accuracy: "
#     f"{results['train_accuracy'].mean():.4f} "
#     f"± {results['train_accuracy'].std():.4f}"
# )

## Experimento F - Combinação de TF-IDF de palavras (1,3) e TF-IDF de caracteres (2,5)

In [ ]:
# # ============================================================
# # EXPERIMENTO F — TF-IDF PALAVRAS + CARACTERES
# # ============================================================

# import pandas as pd
# import numpy as np

# from sklearn.pipeline import Pipeline, FeatureUnion
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.model_selection import StratifiedKFold, cross_validate
# from sklearn.metrics import f1_score, accuracy_score
# from sklearn.model_selection import train_test_split


# # ------------------------------------------------------------
# # 1. Dados
# # ------------------------------------------------------------

# df = pd.read_csv("train_limpo.csv")

# X = df["resp_text"]
# y = df["clarity"]


# # ------------------------------------------------------------
# # 2. Combinação dos vetorizadores
# # ------------------------------------------------------------

# tfidf_union = FeatureUnion([
#     (
#         "word_tfidf",
#         TfidfVectorizer(
#             analyzer="word",
#             sublinear_tf=True,
#             ngram_range=(1, 3)
#         )
#     ),
#     (
#         "char_tfidf",
#         TfidfVectorizer(
#             analyzer="char",
#             sublinear_tf=True,
#             ngram_range=(2, 5)
#         )
#     )
# ])


# # ------------------------------------------------------------
# # 3. Pipeline completo
# # ------------------------------------------------------------

# pipeline = Pipeline([
#     ("features", tfidf_union),
#     (
#         "logreg",
#         LogisticRegression(
#             C=1.0,
#             class_weight="balanced",
#             max_iter=1000
#         )
#     )
# ])

# # ------------------------------------------------------------
# # 4. Avaliação inicial — divisão treino e teste
# # ------------------------------------------------------------

# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.2,
#     stratify=y,
#     random_state=123
# )

# # Treinamento
# pipeline.fit(X_train, y_train)

# # Predições
# y_pred = pipeline.predict(X_test)

# # Métricas
# print("Resultados — Avaliação inicial")
# print(f"Macro F1: {f1_score(y_test, y_pred, average='macro'):.4f}")
# print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")


# # ------------------------------------------------------------
# # 5. Validação cruzada
# # ------------------------------------------------------------

# cv = StratifiedKFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=123
# )

# results = cross_validate(
#     pipeline,
#     X,
#     y,
#     cv=cv,
#     scoring={
#         "f1_macro": "f1_macro",
#         "accuracy": "accuracy"
#     },
#     return_train_score=True,
#     n_jobs=-1
# )


# # ------------------------------------------------------------
# # 6. Resultados por fold
# # ------------------------------------------------------------

# print("Resultados por fold:\n")

# for i in range(5):
#     print(
#         f"Fold {i+1}: "
#         f"F1 Macro = {results['test_f1_macro'][i]:.4f} | "
#         f"Accuracy = {results['test_accuracy'][i]:.4f}"
#     )


# # ------------------------------------------------------------
# # 7. Média e desvio padrão
# # ------------------------------------------------------------

# print("\nResultado da validação cruzada:")

# print(
#     f"Macro F1: "
#     f"{results['test_f1_macro'].mean():.4f} "
#     f"± {results['test_f1_macro'].std():.4f}"
# )

# print(
#     f"Accuracy: "
#     f"{results['test_accuracy'].mean():.4f} "
#     f"± {results['test_accuracy'].std():.4f}"
# )

# print("\nTreino:")

# print(
#     f"Macro F1: "
#     f"{results['train_f1_macro'].mean():.4f} "
#     f"± {results['train_f1_macro'].std():.4f}"
# )

# print(
#     f"Accuracy: "
#     f"{results['train_accuracy'].mean():.4f} "
#     f"± {results['train_accuracy'].std():.4f}"
# )

## GridSearch - Experimento A

In [11]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Pipeline do Modelo A
pipeline_A = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        sublinear_tf=True,
        ngram_range=(1, 3)
    )),
    ("logreg", LogisticRegression(max_iter=1000))
])

# Hiperparâmetros a testar
param_grid_A = {
    "logreg__C": [0.1, 0.5, 1, 2, 5],
    "logreg__class_weight": [None, "balanced"]
}

# Validação cruzada
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

# Grid Search
grid_A = GridSearchCV(
    estimator=pipeline_A,
    param_grid=param_grid_A,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,
    return_train_score=True,
    verbose=1
)

# Treinamento (usando o texto bruto x_train_text em vez de x_train)
grid_A.fit(x_train_text, y_train)

# Resultados
print("Melhores parâmetros:", grid_A.best_params_)
print("Melhor F1 Macro:", grid_A.best_score_)


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Melhores parâmetros: {'logreg__C': 1, 'logreg__class_weight': None}
Melhor F1 Macro: 0.452357850637833


## GridSeacrh - Experimento B (não rodou)

In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Pipeline do Modelo B
pipeline_B = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        sublinear_tf=True,
        ngram_range=(1, 4)
    )),
    ("logreg", LogisticRegression(max_iter=1000))
])

# Hiperparâmetros a testar
param_grid_B = {
    "logreg__C": [0.1, 0.5, 1, 2, 5],
    "logreg__class_weight": [None, "balanced"]
}

# Validação cruzada
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

# Grid Search
grid_B = GridSearchCV(
    estimator=pipeline_B,
    param_grid=param_grid_B,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,
    return_train_score=True,
    verbose=1
)

# Treinamento (usando o texto bruto x_train_text em vez de x_train)
grid_B.fit(x_train_text, y_train)

# Resultados
print("Melhores parâmetros:", grid_B.best_params_)
print("Melhor F1 Macro:", grid_B.best_score_)


Fitting 5 folds for each of 10 candidates, totalling 50 fits


## 6. Treinamento dos novos modelos com todos os dados
Rodei para ngram_range=(1,3) e ngram_range=(1,4), com class_weight=None e class_weight="balanced"

In [15]:
# ============================================================
# TREINAMENTO DO MODELO FINAL COM TODOS OS DADOS ROTULADOS
# ============================================================

# Dados completos de treinamento
X_full = df["resp_text"]
y_full = df["clarity"]

# Novo TF-IDF
vect_final = TfidfVectorizer(
    analyzer="word",
    sublinear_tf=True,
    ngram_range=(1, 4)
)

# Ajustar o TF-IDF usando TODOS os dados rotulados
X_full_tfidf = vect_final.fit_transform(X_full)

print("TF-IDF final:", X_full_tfidf.shape)

# Novo classificador
clf_final = LogisticRegression(
    C=1.0,
    class_weight="balanced", # Trocar aqui - None ou balanced
    max_iter=1000
)

# Treinar usando TODOS os dados rotulados
clf_final.fit(X_full_tfidf, y_full)

print("Modelo final treinado com todos os dados.")

TF-IDF final: (20092, 2582622)
Modelo final treinado com todos os dados.


## 7. Rotulação do test1

In [16]:
# ============================================================
# CLASSIFICAÇÃO DO CONJUNTO DE TESTE 1
# ============================================================

test_df = pd.read_csv("../data/test1_limpo.csv")

X_new = test_df["resp_text"]

# Aplicar o TF-IDF final aos novos textos
X_new_tfidf = vect_final.transform(X_new)

# Fazer as previsões com o modelo final
predicted_test = clf_final.predict(X_new_tfidf)

# Adicionar as previsões
test_df["clarity"] = predicted_test

# Conferir resultado
print(test_df["clarity"].value_counts())
print(test_df.head())

clarity
c5      319
c234    300
c1      281
Name: count, dtype: int64
                                           resp_text clarity
0  prezada hellen dentre as pesquisas domiciliare...    c234
1  senhor francisco o serviço de informações ao c...    c234
2  prezada srª raylane em resposta a sua solicita...      c5
3  prezada senhora de acordo com o art 13 do decr...      c1
4  prezado a senhor a em atendimento ao pedido de...      c1


## 8. Salva o arquivo rotulado

In [17]:
# Salva o arquivo rotulado
test_df.to_excel("../resultados/test1_rotulado_ep2_B.xlsx", index=False)

## 9. Avaliação do novo modelo

In [18]:
# Carregar os arquivos
ideal = pd.read_excel("../data/test1-rotulado-para-teste-estatistico.xlsx")
previsoes = pd.read_excel("../resultados/test1_rotulado_ep2_B.xlsx")

# Conferir se os textos estão na mesma ordem
# assert ideal["resp_text"].equals(previsoes["resp_text"]), \
#     "Os textos não estão na mesma ordem!"

# Rótulos reais e previsões do modelo
y_real = ideal["clarity"]
y_pred = previsoes["clarity"]

classes = ["c1", "c234", "c5"]

# Calcular acurácia
acuracia = accuracy_score(y_real, y_pred)

print(f"Acurácia do novo modelo: {acuracia:.2%}")

# Relatório de classificação
print("\nRelatório de classificação:")
print(classification_report(
    y_real,
    y_pred,
    labels=classes,
    zero_division=0
))

Acurácia do novo modelo: 49.33%

Relatório de classificação:
              precision    recall  f1-score   support

          c1       0.52      0.48      0.50       300
        c234       0.44      0.44      0.44       300
          c5       0.52      0.56      0.54       300

    accuracy                           0.49       900
   macro avg       0.49      0.49      0.49       900
weighted avg       0.49      0.49      0.49       900

